In [0]:
# ============================================================
# 02_silver_transformation
# Healthcare Data Warehouse — Medallion Architecture
# Layer: Silver (Cleaning, Standardization, Deduplication)
# Description: Reads Bronze Delta tables, applies type casting,
#              string standardization, numeric rounding,
#              and deduplication. Writes to Silver container.
# ============================================================
 
# Configuration 

 
storage_account_name = "adlshospitaldwh"
container_landing    = "landing"
container_bronze     = "bronze"
container_silver     = "silver"
container_gold       = "gold"
 
application_id = "ad3497a3-a8be-4251-8e01-4b7bc1c781b4"
tenant_id      = "2be11f7b-4e63-4e64-b8a4-992816fe3359"
client_secret  = "************"
 
# Service Principal OAuth authentication
spark.conf.set(f"fs.azure.account.auth.type.{storage_account_name}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account_name}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account_name}.dfs.core.windows.net", application_id)
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account_name}.dfs.core.windows.net", client_secret)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account_name}.dfs.core.windows.net",
               f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
 
LANDING_PATH = f"abfss://{container_landing}@{storage_account_name}.dfs.core.windows.net/"
BRONZE_PATH  = f"abfss://{container_bronze}@{storage_account_name}.dfs.core.windows.net/"
SILVER_PATH  = f"abfss://{container_silver}@{storage_account_name}.dfs.core.windows.net/"
GOLD_PATH    = f"abfss://{container_gold}@{storage_account_name}.dfs.core.windows.net/"
 
print(" Configuration complete")
print(f"   Bronze : {BRONZE_PATH}")
print(f"   Silver : {SILVER_PATH}")

 Configuration complete
   Bronze : abfss://bronze@adlshospitaldwh.dfs.core.windows.net/
   Silver : abfss://silver@adlshospitaldwh.dfs.core.windows.net/


In [0]:
#  Silver Patients 
from pyspark.sql.functions import col, trim, upper, round, to_date, to_timestamp
 
try:
    df_patients = spark.read.format("delta").load(f"{BRONZE_PATH}patients")
 
    df_patients_silver = (df_patients
        .withColumn("dob",               to_date(col("dob"),               "yyyy-MM-dd"))
        .withColumn("registration_date", to_date(col("registration_date"), "yyyy-MM-dd"))
        .withColumn("gender",            trim(upper(col("gender"))))
        .withColumn("ethnicity",         trim(upper(col("ethnicity"))))
        .withColumn("insurance_type",    trim(upper(col("insurance_type"))))
        .withColumn("marital_status",    trim(upper(col("marital_status"))))
        .dropDuplicates(["patient_id"]))
 
    df_patients_silver.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}patients")
    print(f" patients: {df_patients_silver.count():,} rows")
 
except Exception as e:
    print(f" FAILED — patients: {e}")
    raise
 
 
#  Silver Encounters 
try:
    df_encounters = spark.read.format("delta").load(f"{BRONZE_PATH}encounters")
 
    df_encounters_silver = (df_encounters
        .withColumn("visit_date",      to_date(col("visit_date"),      "yyyy-MM-dd"))
        .withColumn("discharge_date",  to_date(col("discharge_date"),  "yyyy-MM-dd"))
        .withColumn("visit_type",      trim(upper(col("visit_type"))))
        .withColumn("department",      trim(upper(col("department"))))
        .withColumn("admission_type",  trim(upper(col("admission_type"))))
        .withColumn("status",          trim(upper(col("status"))))
        .withColumn("readmitted_flag", trim(upper(col("readmitted_flag"))))
        .dropDuplicates(["encounter_id"]))
 
    df_encounters_silver.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}encounters")
    print(f" encounters: {df_encounters_silver.count():,} rows")
 
except Exception as e:
    print(f" FAILED — encounters: {e}")
    raise
 
 
#  Silver Diagnoses 
# Note: diagnosis_id is NOT a unique key — using full row dedup
try:
    df_diagnoses = spark.read.format("delta").load(f"{BRONZE_PATH}diagnoses")
 
    df_diagnoses_silver = (df_diagnoses
        .withColumn("diagnosis_code",        trim(upper(col("diagnosis_code"))))
        .withColumn("diagnosis_description", trim(col("diagnosis_description")))
        .dropDuplicates())
 
    df_diagnoses_silver.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}diagnoses")
    print(f" diagnoses: {df_diagnoses_silver.count():,} rows")
 
except Exception as e:
    print(f" FAILED — diagnoses: {e}")
    raise
 
 
#  Silver Claims and Billing
# Note: Source date format is DD-MM-YYYY HH:MM (non-standard)
#       Must use inferSchema=false and explicit format string
try:
    df_claims_raw = (spark.read.format("csv")
        .option("header",      "true")
        .option("inferSchema", "false")   # keep dates as strings for explicit parsing
        .load(f"{LANDING_PATH}claims_and_billing.csv"))
 
    df_claims_silver = (df_claims_raw
        .withColumn("claim_billing_date", to_date(to_timestamp(col("claim_billing_date"), "dd-MM-yyyy HH:mm")))
        .withColumn("billed_amount",      round(col("billed_amount").cast("double"), 2))
        .withColumn("paid_amount",        round(col("paid_amount").cast("double"),   2))
        .withColumn("insurance_provider", trim(upper(col("insurance_provider"))))
        .withColumn("payment_method",     trim(upper(col("payment_method"))))
        .withColumn("claim_status",       trim(upper(col("claim_status"))))
        .withColumn("denial_reason",      trim(col("denial_reason")))
        .dropDuplicates(["billing_id"]))
 
    df_claims_silver.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}claims_and_billing")
    print(f" claims_and_billing: {df_claims_silver.count():,} rows")
 
except Exception as e:
    print(f" FAILED — claims_and_billing: {e}")
    raise
 
 
#  Silver Denials 
try:
    df_denials = spark.read.format("delta").load(f"{BRONZE_PATH}denials")
 
    df_denials_silver = (df_denials
        .withColumn("denial_date",             to_date(col("denial_date"),             "yyyy-MM-dd"))
        .withColumn("appeal_resolution_date",  to_date(col("appeal_resolution_date"),  "yyyy-MM-dd"))
        .withColumn("denied_amount",           round(col("denied_amount"), 2))
        .withColumn("denial_reason_code",      trim(upper(col("denial_reason_code"))))
        .withColumn("appeal_filed",            trim(upper(col("appeal_filed"))))
        .withColumn("appeal_status",           trim(upper(col("appeal_status"))))
        .withColumn("final_outcome",           trim(upper(col("final_outcome"))))
        .dropDuplicates(["denial_id"]))
 
    df_denials_silver.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}denials")
    print(f" denials: {df_denials_silver.count():,} rows")
 
except Exception as e:
    print(f" FAILED — denials: {e}")
    raise
 
 
# Silver Procedures 
# Note: procedure_id is NOT a unique key — using full row dedup
try:
    df_procedures = spark.read.format("delta").load(f"{BRONZE_PATH}procedures")
 
    df_procedures_silver = (df_procedures
        .withColumn("procedure_date",        to_date(col("procedure_date"), "yyyy-MM-dd"))
        .withColumn("procedure_code",        trim(upper(col("procedure_code"))))
        .withColumn("procedure_description", trim(col("procedure_description")))
        .withColumn("procedure_cost",        round(col("procedure_cost"), 2))
        .dropDuplicates())
 
    df_procedures_silver.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}procedures")
    print(f" procedures: {df_procedures_silver.count():,} rows")
 
except Exception as e:
    print(f" FAILED — procedures: {e}")
    raise
 
 
#  Silver Medications 
# Note: medication_id is NOT a unique key — using full row dedup
try:
    df_medications = spark.read.format("delta").load(f"{BRONZE_PATH}medications")
 
    df_medications_silver = (df_medications
        .withColumn("prescribed_date", to_date(col("prescribed_date"), "yyyy-MM-dd"))
        .withColumn("drug_name",       trim(upper(col("drug_name"))))
        .withColumn("route",           trim(upper(col("route"))))
        .withColumn("frequency",       trim(upper(col("frequency"))))
        .withColumn("cost",            round(col("cost"), 2))
        .dropDuplicates())   # full row dedup — medication_id is not unique
 
    df_medications_silver.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}medications")
    print(f" medications: {df_medications_silver.count():,} rows")
 
except Exception as e:
    print(f" FAILED — medications: {e}")
    raise
 
 
# Silver Providers 
try:
    df_providers = spark.read.format("delta").load(f"{BRONZE_PATH}providers")
 
    df_providers_silver = (df_providers
        .withColumn("specialty",  trim(upper(col("specialty"))))
        .withColumn("department", trim(upper(col("department"))))
        .withColumn("location",   trim(upper(col("location"))))
        .withColumn("inhouse",    trim(upper(col("inhouse"))))
        .dropDuplicates(["provider_id"]))
 
    df_providers_silver.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}providers")
    print(f" providers: {df_providers_silver.count():,} rows")
 
except Exception as e:
    print(f" FAILED — providers: {e}")
    raise
 
 
# Silver Lab Tests 
# Note: lab_id is NOT a unique key — using full row dedup
try:
    df_lab_tests = spark.read.format("delta").load(f"{BRONZE_PATH}lab_tests")
 
    df_lab_tests_silver = (df_lab_tests
        .withColumn("test_date",     to_date(col("test_date"), "yyyy-MM-dd"))
        .withColumn("test_name",     trim(upper(col("test_name"))))
        .withColumn("test_code",     trim(upper(col("test_code"))))
        .withColumn("specimen_type", trim(upper(col("specimen_type"))))
        .withColumn("status",        trim(upper(col("status"))))
        .dropDuplicates())
 
    df_lab_tests_silver.write.format("delta").mode("overwrite").save(f"{SILVER_PATH}lab_tests")
    print(f" lab_tests: {df_lab_tests_silver.count():,} rows")
 
except Exception as e:
    print(f" FAILED — lab_tests: {e}")
    raise
 
 

 patients: 60,000 rows
 encounters: 70,000 rows
 diagnoses: 70,000 rows
 claims_and_billing: 70,000 rows
 denials: 5,998 rows
 procedures: 126,021 rows
 medications: 94,498 rows
 providers: 1,491 rows
 lab_tests: 54,537 rows


In [0]:
# Verification 
silver_tables = [
    "patients", "encounters", "diagnoses", "claims_and_billing",
    "denials", "procedures", "medications", "providers", "lab_tests"
]
 
expected_counts = {
    "patients":           60_000,
    "encounters":         70_000,
    "diagnoses":          70_000,
    "claims_and_billing": 70_000,
    "denials":             5_998,
    "procedures":        126_021,
    "medications":        94_498,
    "providers":           1_491,
    "lab_tests":          54_537,
}
 
print("\n" + "=" * 50)
print("SILVER LAYER VERIFICATION")
print("=" * 50)
for table in silver_tables:
    try:
        df = spark.read.format("delta").load(f"{SILVER_PATH}{table}")
        actual   = df.count()
        expected = expected_counts[table]
        status   = " " if actual == expected else "  COUNT MISMATCH"
        print(f"{status} {table}: {actual:,} rows | expected {expected:,}")
    except Exception as e:
        print(f" Could not read {table}: {e}")
print("=" * 50)
print(" Silver transformation complete!")


SILVER LAYER VERIFICATION
  patients: 60,000 rows | expected 60,000
  encounters: 70,000 rows | expected 70,000
  diagnoses: 70,000 rows | expected 70,000
  claims_and_billing: 70,000 rows | expected 70,000
  denials: 5,998 rows | expected 5,998
  procedures: 126,021 rows | expected 126,021
  medications: 94,498 rows | expected 94,498
  providers: 1,491 rows | expected 1,491
  lab_tests: 54,537 rows | expected 54,537
 Silver transformation complete!
